# AML Screening of FDA-Approved Small-Molecule Drugs

This notebook applies the trained DAVIS drug–target affinity models to an **exploratory Acute Myeloid Leukemia (AML) drug-repurposing screen**.

The screening data are built from primary/curated sources:

- **AML target selection:** the seven targets discussed in the supplied AML therapeutic-target review
- **Protein sequences:** reviewed human UniProtKB/Swiss-Prot entries retrieved through the UniProt REST API
- **Approved-drug source:** the official FDA Orange Book downloadable data
- **Chemical structures:** PubChem PUG REST
- **Structure validation:** RDKit

Two trained DAVIS models are used:

1. the best-performing DAVIS baseline model on the random test split;
2. the advanced GINE + frozen ESM-2 + CNN + bidirectional cross-attention model.

The output is a computational ranking only. It does **not** establish experimental binding, clinical efficacy, or a new AML indication.

## Notebook workflow

```text
AML target review
      │
      ▼
7 AML targets
BCL2, FLT3, IDH1, IDH2, Menin, CD123, CD47
      │
      ▼
Reviewed human UniProt entries
      │
      ├── accession
      ├── gene
      ├── protein name
      └── canonical protein sequence
      │
      ▼
aml_targets.csv


FDA Orange Book
      │
      ▼
Current RX / OTC approved products
      │
      ▼
Unique active ingredients
      │
      ▼
PubChem PUG REST
      │
      ├── CID
      ├── SMILES
      ├── InChIKey
      ├── molecular formula
      └── molecular weight
      │
      ▼
RDKit structure standardisation
      │
      ▼
candidate_drugs.csv


Candidate drugs × 7 AML targets
              │
              ├─────────────────────────────┐
              ▼                             ▼
 Best DAVIS baseline                 Advanced DAVIS model
 GCN/GAT/GIN + CNN                   GINE + ESM-2 + CNN
                                     + cross-attention
              │                             │
              └──────────────┬──────────────┘
                             ▼
                     Predicted pKd
                             │
                             ▼
                  Model-specific ranks
                             │
                             ▼
                     Consensus ranking
                             │
            ┌────────────────┼─────────────────┐
            ▼                ▼                 ▼
        Top 20 pairs    Top 10/target     Top 20 unique drugs
                             │
                             ▼
                 Save screening outputs
```

The five targets **BCL2, FLT3, IDH1, IDH2 and MEN1** are treated as the main small-molecule screening targets.  
**CD123/IL3RA and CD47** are retained as exploratory targets because the reviewed AML therapeutic strategies for these cell-surface proteins are often antibody- or cell-therapy based.

# Part A — Setup

## 1. Imports

In [ ]:
import gc
import html
import io
import json
import re
import sys
import time
import zipfile

from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, urljoin

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch

from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from requests.adapters import HTTPAdapter
from torch.utils.data import DataLoader as TorchDataLoader
from torch_geometric.loader import DataLoader as PyGDataLoader
from transformers import AutoModel, AutoTokenizer
from urllib3.util.retry import Retry

## 2. Locate the project folder

In [ ]:
def find_project_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/final_aml_gnn_dta"),
    ]

    content_dir = Path("/content")

    if content_dir.exists():
        candidates.extend(
            path.parent
            for path in content_dir.glob("*/config.py")
        )

    for candidate in candidates:
        if (
            (candidate / "config.py").exists()
            and (candidate / "src").exists()
        ):
            return candidate.resolve()

    raise FileNotFoundError(
        "Project root could not be found. "
        "Run this notebook from the final_aml_gnn_dta project."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 3. Project functions and screening settings

The screening uses the **random-split checkpoint** for each selected model because these are the directly comparable DAVIS models used for the main benchmark experiment.

The best baseline architecture is selected automatically from the three saved DAVIS baseline result files using the lowest random-split test RMSE.

In [ ]:
from config import (
    AML_DIR,
    CHECKPOINTS_DIR,
    METRICS_DIR,
    PREDICTIONS_DIR,
    RANDOM_SEED,
    RESULTS_DIR,
)

from src.models import (
    AdvancedDTAModel,
    GATCNN,
    GCNCNN,
    GINCNN,
    count_trainable_parameters,
)

from src.preprocessing import (
    AdvancedInteractionDataset,
    InteractionDataset,
    advanced_collate_fn,
    build_drug_graph_cache,
    build_esm_embedding_cache,
    build_protein_tensor_cache,
    check_esm_embedding_cache,
    load_esm_embedding_cache,
    save_esm_embedding_cache,
)

from src.training import load_checkpoint
from src.utils import get_device, set_seed

set_seed(RANDOM_SEED)
device = get_device()

AML_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

(PREDICTIONS_DIR).mkdir(
    parents=True,
    exist_ok=True,
)

ESM_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
ESM_DIM = 320

SCREENING_SPLIT = "random"

BASELINE_SCREEN_BATCH_SIZE = 64
ADVANCED_SCREEN_BATCH_SIZE = 16
ESM_CACHE_BATCH_SIZE = 7

TOP_N = 20
TOP_PER_TARGET = 10

PUBCHEM_REQUEST_DELAY = 0.25
PUBCHEM_SAVE_EVERY = 50

# Keep None for the final complete screen.
# During a quick connectivity test this may temporarily be changed
# to a small integer, then returned to None for the final run.
MAX_PUBCHEM_LOOKUPS = None

MAX_SCREENING_MW = 1000.0
MIN_HEAVY_ATOMS = 5

RETRIEVAL_DATE = datetime.now(
    timezone.utc
).date().isoformat()

print("Device:", device)
print("Retrieval date:", RETRIEVAL_DATE)
print("Screening split:", SCREENING_SPLIT)

## 4. Official data-source URLs

These URLs are stored directly in the notebook so the provenance of the screening data is explicit and reproducible.

In [ ]:
AML_REVIEW_URL = (
    "https://pmc.ncbi.nlm.nih.gov/articles/"
    "PMC11021231/#Sec3"
)

UNIPROT_API_BASE = (
    "https://rest.uniprot.org/uniprotkb"
)

FDA_ORANGE_BOOK_PAGE = (
    "https://www.fda.gov/drugs/"
    "drug-approvals-and-databases/"
    "orange-book-data-files"
)

FDA_ORANGE_BOOK_ZIP = (
    "https://www.fda.gov/media/76860/download?attachment="
)

PUBCHEM_API_BASE = (
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
)

print("AML review:", AML_REVIEW_URL)
print("UniProt API:", UNIPROT_API_BASE)
print("FDA Orange Book:", FDA_ORANGE_BOOK_PAGE)
print("PubChem PUG REST:", PUBCHEM_API_BASE)

## 5. Create a reusable web session

Retries are used for temporary server errors and rate limiting. PubChem requests are also deliberately spaced to remain below the service's published request-rate guidance.

In [ ]:
retry_policy = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=0.5,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504,
    ],
    allowed_methods=["GET"],
)

session = requests.Session()

adapter = HTTPAdapter(
    max_retries=retry_policy
)

session.mount(
    "https://",
    adapter,
)

session.headers.update(
    {
        "User-Agent": (
            "MSc-AML-GNN-DTA-Dissertation/1.0 "
            "(academic research)"
        )
    }
)

print("Web session ready.")

# Part B — AML targets from the review and UniProt

## 6. Define the seven paper-supported AML targets

The paper target names are mapped to their corresponding human gene symbols and reviewed UniProt accessions.

The UniProt accession is used as the model `target_id` because it provides a stable protein identifier.

In [ ]:
target_reference_df = pd.DataFrame(
    [
        {
            "paper_target": "BCL2",
            "gene": "BCL2",
            "uniprot_id": "P10415",
            "screen_group": "primary",
        },
        {
            "paper_target": "FLT3",
            "gene": "FLT3",
            "uniprot_id": "P36888",
            "screen_group": "primary",
        },
        {
            "paper_target": "IDH1",
            "gene": "IDH1",
            "uniprot_id": "O75874",
            "screen_group": "primary",
        },
        {
            "paper_target": "IDH2",
            "gene": "IDH2",
            "uniprot_id": "P48735",
            "screen_group": "primary",
        },
        {
            "paper_target": "Menin",
            "gene": "MEN1",
            "uniprot_id": "O00255",
            "screen_group": "primary",
        },
        {
            "paper_target": "CD123",
            "gene": "IL3RA",
            "uniprot_id": "P26951",
            "screen_group": "exploratory",
        },
        {
            "paper_target": "CD47",
            "gene": "CD47",
            "uniprot_id": "Q08722",
            "screen_group": "exploratory",
        },
    ]
)

target_reference_df["reference_source"] = (
    "AML therapeutic-target review"
)

target_reference_df["reference_url"] = (
    AML_REVIEW_URL
)

display(target_reference_df)

## 7. Retrieve and verify reviewed human UniProt entries

Each target must satisfy all of the following:

- the returned primary accession matches the expected accession;
- the entry is reviewed/Swiss-Prot;
- the organism is *Homo sapiens* (taxonomy ID 9606);
- the primary gene symbol matches the expected gene;
- a non-empty canonical protein sequence is present.

The notebook stops if any of these checks fail.

In [ ]:
def get_nested_value(
    data,
    keys,
    default=None,
):
    value = data

    for key in keys:
        if not isinstance(value, dict):
            return default

        value = value.get(
            key,
            default,
        )

        if value is default:
            return default

    return value


def fetch_uniprot_target(
    row,
):
    accession = row["uniprot_id"]
    expected_gene = row["gene"]

    url = (
        f"{UNIPROT_API_BASE}/"
        f"{accession}.json"
    )

    response = session.get(
        url,
        timeout=60,
    )
    response.raise_for_status()

    entry = response.json()

    primary_accession = entry.get(
        "primaryAccession"
    )

    entry_type = str(
        entry.get(
            "entryType",
            "",
        )
    )

    organism = entry.get(
        "organism",
        {},
    )

    taxon_id = organism.get(
        "taxonId"
    )

    scientific_name = organism.get(
        "scientificName"
    )

    genes = entry.get(
        "genes",
        [],
    )

    returned_gene = None

    if genes:
        returned_gene = get_nested_value(
            genes[0],
            [
                "geneName",
                "value",
            ],
        )

    protein_name = get_nested_value(
        entry,
        [
            "proteinDescription",
            "recommendedName",
            "fullName",
            "value",
        ],
    )

    if protein_name is None:
        protein_name = get_nested_value(
            entry,
            [
                "proteinDescription",
                "submissionNames",
            ],
            default="",
        )

    sequence_data = entry.get(
        "sequence",
        {},
    )

    protein_sequence = sequence_data.get(
        "value",
        "",
    )

    sequence_length = sequence_data.get(
        "length",
        len(protein_sequence),
    )

    checks = {
        "accession_match": (
            primary_accession == accession
        ),
        "reviewed": (
            "reviewed" in entry_type.lower()
            or "swiss-prot" in entry_type.lower()
        ),
        "human": (
            taxon_id == 9606
        ),
        "gene_match": (
            returned_gene == expected_gene
        ),
        "sequence_present": (
            isinstance(
                protein_sequence,
                str,
            )
            and len(
                protein_sequence
            ) > 0
        ),
    }

    if not all(
        checks.values()
    ):
        raise ValueError(
            f"UniProt verification failed for "
            f"{row['paper_target']} ({accession}): "
            f"{checks}"
        )

    return {
        "paper_target": row[
            "paper_target"
        ],
        "gene": expected_gene,
        "target_id": accession,
        "uniprot_id": accession,
        "uniprot_entry_name": entry.get(
            "uniProtkbId"
        ),
        "protein_name": protein_name,
        "organism": scientific_name,
        "taxonomy_id": taxon_id,
        "reviewed": True,
        "protein_sequence": protein_sequence,
        "sequence_length": int(
            sequence_length
        ),
        "screen_group": row[
            "screen_group"
        ],
        "reference_source": row[
            "reference_source"
        ],
        "reference_url": row[
            "reference_url"
        ],
        "sequence_source": "UniProtKB/Swiss-Prot",
        "sequence_source_url": url,
        "retrieval_date": RETRIEVAL_DATE,
    }

In [ ]:
target_records = []

for _, target_row in (
    target_reference_df.iterrows()
):
    record = fetch_uniprot_target(
        target_row
    )

    target_records.append(
        record
    )

    print(
        f"{record['paper_target']}: "
        f"{record['uniprot_id']} | "
        f"{record['gene']} | "
        f"{record['sequence_length']} aa | "
        "reviewed human entry"
    )

aml_targets_df = pd.DataFrame(
    target_records
)

assert len(
    aml_targets_df
) == 7

assert (
    aml_targets_df["reviewed"]
    .all()
)

assert (
    aml_targets_df[
        "taxonomy_id"
    ]
    .eq(9606)
    .all()
)

display(
    aml_targets_df[
        [
            "paper_target",
            "gene",
            "uniprot_id",
            "protein_name",
            "sequence_length",
            "screen_group",
        ]
    ]
)

## 8. Save the verified AML target table

In [ ]:
AML_TARGETS_PATH = (
    AML_DIR
    / "aml_targets.csv"
)

aml_targets_df.to_csv(
    AML_TARGETS_PATH,
    index=False,
)

print(
    "AML targets saved to:",
    AML_TARGETS_PATH,
)

# Part C — FDA-approved active ingredients

## 9. Download the official FDA Orange Book data

The Orange Book ZIP is downloaded from FDA and read directly in memory.

The code first tries the current FDA ZIP endpoint. If FDA changes the media URL, it falls back to reading the official Orange Book data page and locating the downloadable ZIP link.

In [ ]:
def download_orange_book_zip():
    response = session.get(
        FDA_ORANGE_BOOK_ZIP,
        timeout=120,
    )

    if (
        response.ok
        and response.content[:2] == b"PK"
    ):
        return (
            response.content,
            FDA_ORANGE_BOOK_ZIP,
        )

    page_response = session.get(
        FDA_ORANGE_BOOK_PAGE,
        timeout=60,
    )
    page_response.raise_for_status()

    page_html = html.unescape(
        page_response.text
    )

    candidate_links = re.findall(
        r'href=["\']([^"\']*'
        r'/media/\d+/download'
        r'[^"\']*)["\']',
        page_html,
        flags=re.IGNORECASE,
    )

    for link in candidate_links:
        download_url = urljoin(
            FDA_ORANGE_BOOK_PAGE,
            link,
        )

        candidate_response = session.get(
            download_url,
            timeout=120,
        )

        if (
            candidate_response.ok
            and candidate_response.content[:2]
            == b"PK"
        ):
            return (
                candidate_response.content,
                download_url,
            )

    raise RuntimeError(
        "The FDA Orange Book ZIP could not be downloaded. "
        "Check the FDA Orange Book data page for a changed download URL."
    )


orange_book_bytes, orange_book_download_url = (
    download_orange_book_zip()
)

print(
    "Downloaded Orange Book ZIP:",
    orange_book_download_url,
)

print(
    "ZIP size:",
    f"{len(orange_book_bytes) / 1024 / 1024:.2f} MB",
)

## 10. Read the Orange Book Products table

In [ ]:
def normalise_column_name(
    column_name,
):
    column_name = (
        str(column_name)
        .strip()
        .lower()
    )

    column_name = re.sub(
        r"[^a-z0-9]+",
        "_",
        column_name,
    )

    return column_name.strip(
        "_"
    )


with zipfile.ZipFile(
    io.BytesIO(
        orange_book_bytes
    )
) as archive:
    product_files = [
        name
        for name in archive.namelist()
        if Path(
            name
        ).name.lower()
        == "products.txt"
    ]

    if len(product_files) != 1:
        raise ValueError(
            "Expected exactly one Products.txt "
            "inside the Orange Book ZIP."
        )

    product_file = product_files[0]

    product_bytes = archive.read(
        product_file
    )

orange_book_df = pd.read_csv(
    io.BytesIO(
        product_bytes
    ),
    sep="~",
    dtype=str,
    encoding="latin1",
    keep_default_na=False,
)

orange_book_df.columns = [
    normalise_column_name(
        column
    )
    for column in orange_book_df.columns
]

print(
    "Orange Book product rows:",
    len(orange_book_df),
)

print(
    "Columns:",
    list(
        orange_book_df.columns
    ),
)

display(
    orange_book_df.head()
)

## 11. Extract current RX/OTC active ingredients

Orange Book rows classified as `DISCN` are excluded from the screening library.

Combination products are split into their individual active ingredients and duplicated ingredient names are removed.

In [ ]:
ingredient_column = next(
    (
        column
        for column
        in orange_book_df.columns
        if column == "ingredient"
        or column.startswith(
            "ingredient_"
        )
    ),
    None,
)

if ingredient_column is None:
    raise KeyError(
        "Could not find the Orange Book Ingredient column."
    )

type_column = (
    "type"
    if "type" in orange_book_df.columns
    else None
)

current_products_df = (
    orange_book_df.copy()
)

if type_column is not None:
    current_products_df = (
        current_products_df[
            current_products_df[
                type_column
            ]
            .str.upper()
            .isin(
                [
                    "RX",
                    "OTC",
                ]
            )
        ]
        .copy()
    )

ingredient_series = (
    current_products_df[
        ingredient_column
    ]
    .astype(str)
    .str.split(";")
    .explode()
    .str.strip()
)

ingredient_series = (
    ingredient_series[
        ingredient_series.ne("")
        & ingredient_series.ne(
            "N/A"
        )
    ]
)

fda_ingredients_df = pd.DataFrame(
    {
        "fda_ingredient": sorted(
            ingredient_series.unique()
        )
    }
)

fda_ingredients_df[
    "approval_source"
] = "FDA Orange Book"

fda_ingredients_df[
    "approval_source_url"
] = FDA_ORANGE_BOOK_PAGE

fda_ingredients_df[
    "fda_download_url"
] = orange_book_download_url

fda_ingredients_df[
    "retrieval_date"
] = RETRIEVAL_DATE

print(
    "Current RX/OTC product rows:",
    len(current_products_df),
)

print(
    "Unique active ingredients:",
    len(fda_ingredients_df),
)

display(
    fda_ingredients_df.head(20)
)

## 12. Save the FDA active-ingredient source table

In [ ]:
FDA_INGREDIENTS_PATH = (
    AML_DIR
    / "fda_active_ingredients.csv"
)

fda_ingredients_df.to_csv(
    FDA_INGREDIENTS_PATH,
    index=False,
)

print(
    "FDA active ingredients saved to:",
    FDA_INGREDIENTS_PATH,
)

# Part D — PubChem structures and RDKit validation

## 13. PubChem lookup cache

Each FDA active ingredient is resolved against PubChem PUG REST.

The lookup records:

- PubChem CID
- PubChem SMILES
- InChIKey
- molecular formula
- PubChem molecular weight
- lookup status

Results are saved incrementally so an interrupted Colab session can resume without repeating completed PubChem requests.

**Important:** the complete first run can take several minutes because requests are intentionally rate-limited. Later runs use the local cache.

In [ ]:
PUBCHEM_CACHE_PATH = (
    AML_DIR
    / "pubchem_fda_lookup.csv"
)


def fetch_pubchem_properties(
    ingredient_name,
):
    encoded_name = quote(
        ingredient_name,
        safe="",
    )

    properties = (
        "SMILES,"
        "InChIKey,"
        "MolecularFormula,"
        "MolecularWeight"
    )

    url = (
        f"{PUBCHEM_API_BASE}/"
        f"compound/name/"
        f"{encoded_name}/"
        f"property/{properties}/JSON"
    )

    response = session.get(
        url,
        timeout=60,
    )

    if response.status_code == 404:
        return {
            "fda_ingredient": ingredient_name,
            "pubchem_status": "not_found",
            "pubchem_cid": None,
            "pubchem_smiles": None,
            "inchikey": None,
            "molecular_formula": None,
            "pubchem_molecular_weight": None,
            "pubchem_source_url": url,
        }

    response.raise_for_status()

    payload = response.json()

    properties_list = (
        payload
        .get(
            "PropertyTable",
            {},
        )
        .get(
            "Properties",
            [],
        )
    )

    if not properties_list:
        return {
            "fda_ingredient": ingredient_name,
            "pubchem_status": "no_properties",
            "pubchem_cid": None,
            "pubchem_smiles": None,
            "inchikey": None,
            "molecular_formula": None,
            "pubchem_molecular_weight": None,
            "pubchem_source_url": url,
        }

    compound = properties_list[0]

    smiles = (
        compound.get("SMILES")
        or compound.get(
            "ConnectivitySMILES"
        )
        or compound.get(
            "CanonicalSMILES"
        )
        or compound.get(
            "IsomericSMILES"
        )
    )

    return {
        "fda_ingredient": ingredient_name,
        "pubchem_status": "resolved",
        "pubchem_cid": compound.get(
            "CID"
        ),
        "pubchem_smiles": smiles,
        "inchikey": compound.get(
            "InChIKey"
        ),
        "molecular_formula": compound.get(
            "MolecularFormula"
        ),
        "pubchem_molecular_weight": compound.get(
            "MolecularWeight"
        ),
        "pubchem_source_url": url,
    }

## 14. Run or resume PubChem enrichment

In [ ]:
if PUBCHEM_CACHE_PATH.exists():
    pubchem_cache_df = pd.read_csv(
        PUBCHEM_CACHE_PATH,
        dtype={
            "fda_ingredient": str,
        },
    )

    print(
        "Existing PubChem cache rows:",
        len(pubchem_cache_df),
    )

else:
    pubchem_cache_df = pd.DataFrame(
        columns=[
            "fda_ingredient",
            "pubchem_status",
            "pubchem_cid",
            "pubchem_smiles",
            "inchikey",
            "molecular_formula",
            "pubchem_molecular_weight",
            "pubchem_source_url",
            "retrieval_date",
        ]
    )

successful_cache_rows = (
    pubchem_cache_df[
        pubchem_cache_df[
            "pubchem_status"
        ].isin(
            [
                "resolved",
                "not_found",
                "no_properties",
            ]
        )
    ]
)

completed_ingredients = set(
    successful_cache_rows[
        "fda_ingredient"
    ]
    .dropna()
    .astype(str)
)

remaining_ingredients = [
    ingredient
    for ingredient
    in fda_ingredients_df[
        "fda_ingredient"
    ].tolist()
    if ingredient
    not in completed_ingredients
]

if MAX_PUBCHEM_LOOKUPS is not None:
    remaining_ingredients = (
        remaining_ingredients[
            :MAX_PUBCHEM_LOOKUPS
        ]
    )

print(
    "Remaining PubChem lookups:",
    len(remaining_ingredients),
)

In [ ]:
new_records = []

for number, ingredient in enumerate(
    remaining_ingredients,
    start=1,
):
    try:
        record = fetch_pubchem_properties(
            ingredient
        )

    except requests.RequestException as error:
        record = {
            "fda_ingredient": ingredient,
            "pubchem_status": (
                "request_error"
            ),
            "pubchem_cid": None,
            "pubchem_smiles": None,
            "inchikey": None,
            "molecular_formula": None,
            "pubchem_molecular_weight": None,
            "pubchem_source_url": None,
            "error": str(error),
        }

    record[
        "retrieval_date"
    ] = RETRIEVAL_DATE

    new_records.append(
        record
    )

    if (
        number % PUBCHEM_SAVE_EVERY == 0
        or number
        == len(
            remaining_ingredients
        )
    ):
        new_df = pd.DataFrame(
            new_records
        )

        pubchem_cache_df = pd.concat(
            [
                pubchem_cache_df,
                new_df,
            ],
            ignore_index=True,
        )

        pubchem_cache_df = (
            pubchem_cache_df
            .drop_duplicates(
                subset=[
                    "fda_ingredient"
                ],
                keep="last",
            )
            .sort_values(
                "fda_ingredient"
            )
            .reset_index(
                drop=True
            )
        )

        pubchem_cache_df.to_csv(
            PUBCHEM_CACHE_PATH,
            index=False,
        )

        new_records = []

        print(
            f"PubChem progress: "
            f"{number}/"
            f"{len(remaining_ingredients)} | "
            f"cache rows: "
            f"{len(pubchem_cache_df)}"
        )

    time.sleep(
        PUBCHEM_REQUEST_DELAY
    )

print(
    "PubChem cache saved to:",
    PUBCHEM_CACHE_PATH,
)

## 15. Check PubChem resolution

In [ ]:
pubchem_summary = (
    pubchem_cache_df[
        "pubchem_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "status"
    )
    .reset_index(
        name="count"
    )
)

display(pubchem_summary)

resolved_pubchem_df = (
    pubchem_cache_df[
        pubchem_cache_df[
            "pubchem_status"
        ]
        == "resolved"
    ]
    .copy()
)

print(
    "Resolved active ingredients:",
    len(resolved_pubchem_df),
)

## 16. Standardise and validate PubChem structures with RDKit

The PubChem structure is converted to an RDKit molecule and reduced to its fragment parent. This removes common counter-ion fragments before graph construction.

A compound is retained for this small-molecule screen when:

- RDKit can parse and standardise the structure;
- at least 5 heavy atoms remain;
- RDKit molecular weight is at most 1000 Da.

These filters are used to keep the screening library reasonably aligned with the molecular-graph domain used by the DTA models.

In [ ]:
def standardise_candidate_smiles(
    smiles,
):
    result = {
        "rdkit_valid": False,
        "screening_smiles": None,
        "rdkit_molecular_weight": None,
        "heavy_atom_count": None,
        "has_multiple_fragments": None,
        "structure_note": None,
    }

    if (
        not isinstance(
            smiles,
            str,
        )
        or not smiles.strip()
    ):
        result[
            "structure_note"
        ] = "missing_smiles"

        return result

    try:
        molecule = Chem.MolFromSmiles(
            smiles
        )

        if molecule is None:
            result[
                "structure_note"
            ] = "rdkit_parse_failed"

            return result

        fragment_count = len(
            Chem.GetMolFrags(
                molecule
            )
        )

        parent = (
            rdMolStandardize
            .FragmentParent(
                molecule
            )
        )

        if (
            parent is None
            or parent.GetNumAtoms()
            == 0
        ):
            result[
                "structure_note"
            ] = "fragment_parent_failed"

            return result

        screening_smiles = (
            Chem.MolToSmiles(
                parent,
                canonical=True,
                isomericSmiles=True,
            )
        )

        result.update(
            {
                "rdkit_valid": True,
                "screening_smiles": screening_smiles,
                "rdkit_molecular_weight": float(
                    Descriptors.MolWt(
                        parent
                    )
                ),
                "heavy_atom_count": int(
                    parent.GetNumHeavyAtoms()
                ),
                "has_multiple_fragments": (
                    fragment_count > 1
                ),
                "structure_note": "valid",
            }
        )

    except Exception as error:
        result[
            "structure_note"
        ] = (
            f"rdkit_error: "
            f"{type(error).__name__}"
        )

    return result

In [ ]:
structure_records = [
    standardise_candidate_smiles(
        smiles
    )
    for smiles
    in resolved_pubchem_df[
        "pubchem_smiles"
    ]
]

structure_df = pd.DataFrame(
    structure_records,
    index=resolved_pubchem_df.index,
)

candidate_audit_df = pd.concat(
    [
        resolved_pubchem_df,
        structure_df,
    ],
    axis=1,
)

candidate_audit_df[
    "eligible_for_screening"
] = (
    candidate_audit_df[
        "rdkit_valid"
    ]
    & (
        candidate_audit_df[
            "heavy_atom_count"
        ]
        >= MIN_HEAVY_ATOMS
    )
    & (
        candidate_audit_df[
            "rdkit_molecular_weight"
        ]
        <= MAX_SCREENING_MW
    )
)

print(
    "RDKit-valid structures:",
    int(
        candidate_audit_df[
            "rdkit_valid"
        ].sum()
    ),
)

print(
    "Eligible structures:",
    int(
        candidate_audit_df[
            "eligible_for_screening"
        ].sum()
    ),
)

## 17. Build the final approved-drug screening library

Different FDA ingredient names can resolve to the same standardised chemical structure. The final library therefore removes duplicate structures after RDKit standardisation.

The original FDA ingredient name and PubChem identifier are retained for provenance.

In [ ]:
candidate_drugs_df = (
    candidate_audit_df[
        candidate_audit_df[
            "eligible_for_screening"
        ]
    ]
    .copy()
)

candidate_drugs_df = (
    candidate_drugs_df
    .sort_values(
        [
            "screening_smiles",
            "fda_ingredient",
        ]
    )
    .drop_duplicates(
        subset=[
            "screening_smiles"
        ],
        keep="first",
    )
    .reset_index(
        drop=True
    )
)

candidate_drugs_df[
    "pubchem_cid"
] = pd.to_numeric(
    candidate_drugs_df[
        "pubchem_cid"
    ],
    errors="coerce",
).astype(
    "Int64"
)

candidate_drugs_df[
    "drug_id"
] = (
    "PUBCHEM_"
    + candidate_drugs_df[
        "pubchem_cid"
    ].astype(str)
)

candidate_drugs_df[
    "drug_name"
] = candidate_drugs_df[
    "fda_ingredient"
]

candidate_drugs_df[
    "smiles"
] = candidate_drugs_df[
    "screening_smiles"
]

candidate_drugs_df[
    "approval_source"
] = "FDA Orange Book"

candidate_drugs_df[
    "structure_source"
] = "PubChem PUG REST"

candidate_drugs_df[
    "rdkit_valid"
] = True

candidate_columns = [
    "drug_id",
    "drug_name",
    "pubchem_cid",
    "smiles",
    "inchikey",
    "molecular_formula",
    "pubchem_molecular_weight",
    "rdkit_molecular_weight",
    "heavy_atom_count",
    "approval_source",
    "approval_source_url",
    "structure_source",
    "pubchem_source_url",
    "retrieval_date",
]

candidate_drugs_df = (
    candidate_drugs_df[
        candidate_columns
    ]
)

CANDIDATE_DRUGS_PATH = (
    AML_DIR
    / "candidate_drugs.csv"
)

candidate_drugs_df.to_csv(
    CANDIDATE_DRUGS_PATH,
    index=False,
)

print(
    "Final unique approved-drug structures:",
    len(candidate_drugs_df),
)

print(
    "Candidate drug file:",
    CANDIDATE_DRUGS_PATH,
)

display(
    candidate_drugs_df.head(20)
)

## 18. Save the structure-screening audit

The audit file preserves resolved PubChem compounds that were excluded by RDKit or the small-molecule filters.

In [ ]:
CANDIDATE_AUDIT_PATH = (
    AML_DIR
    / "candidate_drug_structure_audit.csv"
)

candidate_audit_df.to_csv(
    CANDIDATE_AUDIT_PATH,
    index=False,
)

print(
    "Structure audit saved to:",
    CANDIDATE_AUDIT_PATH,
)

# Part E — Select the trained models

## 19. Select the best DAVIS baseline

The baseline architecture is selected from the three DAVIS baseline result files using **random-split test RMSE**.

The advanced model is screened separately regardless of whether it outperformed the selected baseline.

In [ ]:
BASELINE_METRIC_FILES = {
    "GCN + CNN": (
        METRICS_DIR
        / "davis_gcn_cnn.csv"
    ),
    "GAT + CNN": (
        METRICS_DIR
        / "davis_gat_cnn.csv"
    ),
    "GIN + CNN": (
        METRICS_DIR
        / "davis_gin_cnn.csv"
    ),
}

missing_baseline_metrics = [
    path.name
    for path
    in BASELINE_METRIC_FILES.values()
    if not path.exists()
]

if missing_baseline_metrics:
    raise FileNotFoundError(
        "Missing DAVIS baseline metric files: "
        + ", ".join(
            missing_baseline_metrics
        )
    )

baseline_rows = []

for model_name, file_path in (
    BASELINE_METRIC_FILES.items()
):
    result_df = pd.read_csv(
        file_path
    )

    row = result_df[
        result_df["split"]
        == SCREENING_SPLIT
    ]

    if len(row) != 1:
        raise ValueError(
            f"{file_path.name} must contain one "
            f"{SCREENING_SPLIT} result row."
        )

    row = row.iloc[0].copy()
    row["model"] = model_name
    baseline_rows.append(
        row
    )

baseline_selection_df = pd.DataFrame(
    baseline_rows
)

baseline_selection_df = (
    baseline_selection_df
    .sort_values(
        "test_rmse"
    )
    .reset_index(
        drop=True
    )
)

display(
    baseline_selection_df[
        [
            "model",
            "test_rmse",
            "test_mae",
            "ci",
            "pearson",
        ]
    ].round(4)
)

selected_baseline_name = (
    baseline_selection_df
    .iloc[0]["model"]
)

print(
    "Selected baseline:",
    selected_baseline_name,
)

## 20. Load the selected baseline checkpoint

In [ ]:
BASELINE_MODEL_CLASSES = {
    "GCN + CNN": GCNCNN,
    "GAT + CNN": GATCNN,
    "GIN + CNN": GINCNN,
}

BASELINE_CHECKPOINT_NAMES = {
    "GCN + CNN": (
        "davis_gcn_cnn_random.pt"
    ),
    "GAT + CNN": (
        "davis_gat_cnn_random.pt"
    ),
    "GIN + CNN": (
        "davis_gin_cnn_random.pt"
    ),
}

baseline_checkpoint_path = (
    CHECKPOINTS_DIR
    / BASELINE_CHECKPOINT_NAMES[
        selected_baseline_name
    ]
)

baseline_model = (
    BASELINE_MODEL_CLASSES[
        selected_baseline_name
    ]()
)

load_checkpoint(
    model=baseline_model,
    checkpoint_path=baseline_checkpoint_path,
    device=device,
)

baseline_model.eval()

print(
    "Baseline checkpoint:",
    baseline_checkpoint_path,
)

print(
    "Baseline trainable parameters:",
    f"{count_trainable_parameters(baseline_model):,}",
)

## 21. Load the advanced-model checkpoint

In [ ]:
advanced_metric_path = (
    METRICS_DIR
    / "davis_advanced_gine_esm2.csv"
)

advanced_checkpoint_path = (
    CHECKPOINTS_DIR
    / "davis_advanced_gine_esm2_random.pt"
)

if not advanced_metric_path.exists():
    raise FileNotFoundError(
        f"Missing advanced metrics: "
        f"{advanced_metric_path}"
    )

if not advanced_checkpoint_path.exists():
    raise FileNotFoundError(
        f"Missing advanced checkpoint: "
        f"{advanced_checkpoint_path}"
    )

advanced_model = AdvancedDTAModel(
    esm_dim=ESM_DIM,
    hidden_dim=128,
    attention_heads=4,
    dropout=0.20,
    regressor_dropout=0.30,
)

load_checkpoint(
    model=advanced_model,
    checkpoint_path=advanced_checkpoint_path,
    device=device,
)

advanced_model.eval()

print(
    "Advanced checkpoint:",
    advanced_checkpoint_path,
)

print(
    "Advanced trainable parameters:",
    f"{count_trainable_parameters(advanced_model):,}",
)

# Part F — Create all approved-drug × AML-target pairs

## 22. Build the screening matrix

Every eligible approved drug is paired with every one of the seven AML targets.

The `affinity` column is set to zero only because the existing dataset classes require a label field. These placeholder values are **never used as experimental labels or evaluation targets** during screening.

In [ ]:
drug_screen_df = (
    candidate_drugs_df[
        [
            "drug_id",
            "drug_name",
            "pubchem_cid",
            "smiles",
        ]
    ]
    .copy()
)

target_screen_df = (
    aml_targets_df[
        [
            "target_id",
            "paper_target",
            "gene",
            "protein_sequence",
            "screen_group",
        ]
    ]
    .copy()
)

screening_pairs_df = drug_screen_df.merge(
    target_screen_df,
    how="cross",
)

screening_pairs_df[
    "affinity"
] = 0.0

print(
    "Approved drugs:",
    screening_pairs_df[
        "drug_id"
    ].nunique(),
)

print(
    "AML targets:",
    screening_pairs_df[
        "target_id"
    ].nunique(),
)

print(
    "Total drug-target pairs:",
    len(screening_pairs_df),
)

expected_pairs = (
    len(candidate_drugs_df)
    * len(aml_targets_df)
)

assert (
    len(screening_pairs_df)
    == expected_pairs
)

display(
    screening_pairs_df.head()
)

## 23. Build molecular graphs and baseline protein tensors

In [ ]:
screening_drug_graph_cache = (
    build_drug_graph_cache(
        screening_pairs_df
    )
)

screening_protein_tensor_cache = (
    build_protein_tensor_cache(
        screening_pairs_df
    )
)

print(
    "Drug graphs:",
    len(
        screening_drug_graph_cache
    ),
)

print(
    "Baseline protein tensors:",
    len(
        screening_protein_tensor_cache
    ),
)

# Part G — Best-baseline AML screening

## 24. Predict AML affinities with the selected baseline

In [ ]:
baseline_screen_dataset = InteractionDataset(
    dataframe=screening_pairs_df,
    drug_graph_cache=screening_drug_graph_cache,
    protein_tensor_cache=screening_protein_tensor_cache,
)

baseline_screen_loader = PyGDataLoader(
    baseline_screen_dataset,
    batch_size=BASELINE_SCREEN_BATCH_SIZE,
    shuffle=False,
)


@torch.no_grad()
def predict_baseline_screen(
    model,
    data_loader,
    device,
):
    model.eval()

    predictions = []

    for batch in data_loader:
        batch = batch.to(
            device
        )

        batch_predictions = model(
            batch
        )

        predictions.append(
            batch_predictions
            .detach()
            .cpu()
            .numpy()
        )

    return np.concatenate(
        predictions
    )


baseline_predictions = (
    predict_baseline_screen(
        model=baseline_model,
        data_loader=baseline_screen_loader,
        device=device,
    )
)

assert (
    len(baseline_predictions)
    == len(screening_pairs_df)
)

print(
    "Baseline predictions:",
    len(baseline_predictions),
)

print(
    "Predicted pKd range:",
    f"{baseline_predictions.min():.3f}",
    "to",
    f"{baseline_predictions.max():.3f}",
)

# Part H — Advanced-model AML screening

## 25. Create or load frozen ESM-2 embeddings for the seven AML targets

Only seven protein sequences are embedded, and the resulting residue embeddings are cached for future notebook runs.

In [ ]:
AML_ESM_CACHE_DIR = (
    RESULTS_DIR
    / "esm_embeddings"
)

AML_ESM_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

AML_ESM_CACHE_PATH = (
    AML_ESM_CACHE_DIR
    / "aml_7_targets_esm2_t6_embeddings.pt"
)

if AML_ESM_CACHE_PATH.exists():
    print(
        "Loading existing AML ESM-2 cache..."
    )

    aml_esm_cache = (
        load_esm_embedding_cache(
            AML_ESM_CACHE_PATH
        )
    )

else:
    print(
        "Creating frozen ESM-2 embeddings "
        "for the seven AML targets..."
    )

    esm_device = (
        torch.device("cuda")
        if torch.cuda.is_available()
        else torch.device("cpu")
    )

    tokenizer = (
        AutoTokenizer.from_pretrained(
            ESM_MODEL_NAME
        )
    )

    esm_model = (
        AutoModel.from_pretrained(
            ESM_MODEL_NAME
        )
    )

    aml_esm_cache = (
        build_esm_embedding_cache(
            dataframe=aml_targets_df,
            tokenizer=tokenizer,
            esm_model=esm_model,
            device=esm_device,
            target_id_column="target_id",
            sequence_column="protein_sequence",
            batch_size=ESM_CACHE_BATCH_SIZE,
            storage_dtype=torch.float16,
        )
    )

    save_esm_embedding_cache(
        aml_esm_cache,
        AML_ESM_CACHE_PATH,
    )

    del esm_model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

cache_valid = (
    check_esm_embedding_cache(
        dataframe=aml_targets_df,
        embedding_cache=aml_esm_cache,
        target_id_column="target_id",
    )
)

assert cache_valid

print(
    "AML ESM-2 cache valid:",
    cache_valid,
)

print(
    "Cached targets:",
    len(aml_esm_cache),
)

## 26. Predict AML affinities with the advanced model

In [ ]:
advanced_screen_dataset = (
    AdvancedInteractionDataset(
        dataframe=screening_pairs_df,
        drug_graph_cache=screening_drug_graph_cache,
        protein_embedding_cache=aml_esm_cache,
    )
)

advanced_screen_loader = TorchDataLoader(
    advanced_screen_dataset,
    batch_size=ADVANCED_SCREEN_BATCH_SIZE,
    shuffle=False,
    collate_fn=advanced_collate_fn,
    num_workers=0,
)


@torch.no_grad()
def predict_advanced_screen(
    model,
    data_loader,
    device,
):
    model.eval()

    predictions = []

    for batch in data_loader:
        graph = batch[
            "graph"
        ].to(
            device
        )

        protein_embeddings = batch[
            "protein_embeddings"
        ].to(
            device
        )

        protein_mask = batch[
            "protein_mask"
        ].to(
            device
        )

        batch_predictions = model(
            graph,
            protein_embeddings,
            protein_mask,
        )

        predictions.append(
            batch_predictions
            .detach()
            .cpu()
            .numpy()
        )

    return np.concatenate(
        predictions
    )


advanced_predictions = (
    predict_advanced_screen(
        model=advanced_model,
        data_loader=advanced_screen_loader,
        device=device,
    )
)

assert (
    len(advanced_predictions)
    == len(screening_pairs_df)
)

print(
    "Advanced predictions:",
    len(advanced_predictions),
)

print(
    "Predicted pKd range:",
    f"{advanced_predictions.min():.3f}",
    "to",
    f"{advanced_predictions.max():.3f}",
)

# Part I — Ranking and consensus analysis

## 27. Combine predictions

Both models were trained on DAVIS pKd, so higher predictions correspond to stronger predicted affinity on the model's learned DAVIS scale.

The models can have different calibration, so the main consensus score uses **ranks** rather than treating the mean predicted pKd as experimental affinity.

In [ ]:
screening_results_df = (
    screening_pairs_df.copy()
)

screening_results_df[
    "baseline_model"
] = selected_baseline_name

screening_results_df[
    "baseline_predicted_pkd"
] = baseline_predictions

screening_results_df[
    "advanced_model"
] = (
    "GINE + frozen ESM-2 + CNN "
    "+ cross-attention"
)

screening_results_df[
    "advanced_predicted_pkd"
] = advanced_predictions

screening_results_df[
    "mean_model_predicted_pkd"
] = (
    screening_results_df[
        "baseline_predicted_pkd"
    ]
    + screening_results_df[
        "advanced_predicted_pkd"
    ]
) / 2.0

display(
    screening_results_df[
        [
            "drug_name",
            "paper_target",
            "screen_group",
            "baseline_predicted_pkd",
            "advanced_predicted_pkd",
            "mean_model_predicted_pkd",
        ]
    ].head()
)

## 28. Calculate global and target-specific ranks

In [ ]:
screening_results_df[
    "baseline_global_rank"
] = (
    screening_results_df[
        "baseline_predicted_pkd"
    ]
    .rank(
        method="min",
        ascending=False,
    )
)

screening_results_df[
    "advanced_global_rank"
] = (
    screening_results_df[
        "advanced_predicted_pkd"
    ]
    .rank(
        method="min",
        ascending=False,
    )
)

screening_results_df[
    "consensus_global_rank_score"
] = (
    screening_results_df[
        "baseline_global_rank"
    ]
    + screening_results_df[
        "advanced_global_rank"
    ]
) / 2.0

screening_results_df[
    "baseline_target_rank"
] = (
    screening_results_df
    .groupby(
        "target_id"
    )[
        "baseline_predicted_pkd"
    ]
    .rank(
        method="min",
        ascending=False,
    )
)

screening_results_df[
    "advanced_target_rank"
] = (
    screening_results_df
    .groupby(
        "target_id"
    )[
        "advanced_predicted_pkd"
    ]
    .rank(
        method="min",
        ascending=False,
    )
)

screening_results_df[
    "consensus_target_rank_score"
] = (
    screening_results_df[
        "baseline_target_rank"
    ]
    + screening_results_df[
        "advanced_target_rank"
    ]
) / 2.0

screening_results_df[
    "consensus_target_rank"
] = (
    screening_results_df
    .groupby(
        "target_id"
    )[
        "consensus_target_rank_score"
    ]
    .rank(
        method="first",
        ascending=True,
    )
)

screening_results_df = (
    screening_results_df
    .sort_values(
        [
            "consensus_global_rank_score",
            "mean_model_predicted_pkd",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

screening_results_df[
    "consensus_global_rank"
] = np.arange(
    1,
    len(
        screening_results_df
    ) + 1,
)

display(
    screening_results_df[
        [
            "consensus_global_rank",
            "drug_name",
            "paper_target",
            "screen_group",
            "baseline_predicted_pkd",
            "advanced_predicted_pkd",
            "consensus_global_rank_score",
        ]
    ].head(TOP_N)
)

## 29. Save all AML screening predictions

In [ ]:
ALL_SCREENING_PATH = (
    PREDICTIONS_DIR
    / "aml_all_screening_predictions.csv"
)

screening_results_df.to_csv(
    ALL_SCREENING_PATH,
    index=False,
)

print(
    "All AML predictions saved to:",
    ALL_SCREENING_PATH,
)

## 30. Top 20 baseline and advanced drug-target pairs

In [ ]:
baseline_top_df = (
    screening_results_df
    .sort_values(
        "baseline_predicted_pkd",
        ascending=False,
    )
    .head(TOP_N)
    .copy()
)

advanced_top_df = (
    screening_results_df
    .sort_values(
        "advanced_predicted_pkd",
        ascending=False,
    )
    .head(TOP_N)
    .copy()
)

BASELINE_TOP_PATH = (
    PREDICTIONS_DIR
    / "aml_baseline_top20_pairs.csv"
)

ADVANCED_TOP_PATH = (
    PREDICTIONS_DIR
    / "aml_advanced_top20_pairs.csv"
)

baseline_top_df.to_csv(
    BASELINE_TOP_PATH,
    index=False,
)

advanced_top_df.to_csv(
    ADVANCED_TOP_PATH,
    index=False,
)

print(
    "Top baseline model:",
    selected_baseline_name,
)

display(
    baseline_top_df[
        [
            "drug_name",
            "paper_target",
            "baseline_predicted_pkd",
        ]
    ]
)

print()
print("Top advanced model predictions")

display(
    advanced_top_df[
        [
            "drug_name",
            "paper_target",
            "advanced_predicted_pkd",
        ]
    ]
)

## 31. Top 20 consensus drug-target pairs

In [ ]:
consensus_top_df = (
    screening_results_df
    .head(TOP_N)
    .copy()
)

CONSENSUS_TOP_PATH = (
    PREDICTIONS_DIR
    / "aml_consensus_top20_pairs.csv"
)

consensus_top_df.to_csv(
    CONSENSUS_TOP_PATH,
    index=False,
)

display(
    consensus_top_df[
        [
            "consensus_global_rank",
            "drug_name",
            "paper_target",
            "screen_group",
            "baseline_predicted_pkd",
            "advanced_predicted_pkd",
            "consensus_global_rank_score",
        ]
    ].round(4)
)

print(
    "Consensus Top-20 saved to:",
    CONSENSUS_TOP_PATH,
)

## 32. Top 20 consensus pairs restricted to the five primary small-molecule targets

In [ ]:
primary_results_df = (
    screening_results_df[
        screening_results_df[
            "screen_group"
        ]
        == "primary"
    ]
    .copy()
)

primary_results_df[
    "primary_consensus_rank"
] = (
    primary_results_df[
        "consensus_global_rank_score"
    ]
    .rank(
        method="min",
        ascending=True,
    )
)

primary_top_df = (
    primary_results_df
    .sort_values(
        [
            "primary_consensus_rank",
            "mean_model_predicted_pkd",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .head(TOP_N)
    .copy()
)

PRIMARY_TOP_PATH = (
    PREDICTIONS_DIR
    / "aml_primary_targets_consensus_top20.csv"
)

primary_top_df.to_csv(
    PRIMARY_TOP_PATH,
    index=False,
)

display(
    primary_top_df[
        [
            "primary_consensus_rank",
            "drug_name",
            "paper_target",
            "baseline_predicted_pkd",
            "advanced_predicted_pkd",
            "consensus_global_rank_score",
        ]
    ].round(4)
)

## 33. Top 10 candidates for each AML target

This table uses consensus ranks **within each target**, which avoids a target with generally higher predicted values dominating every target-specific list.

In [ ]:
top_per_target_df = (
    screening_results_df[
        screening_results_df[
            "consensus_target_rank"
        ]
        <= TOP_PER_TARGET
    ]
    .sort_values(
        [
            "paper_target",
            "consensus_target_rank_score",
            "mean_model_predicted_pkd",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

TOP_PER_TARGET_PATH = (
    PREDICTIONS_DIR
    / "aml_top10_per_target.csv"
)

top_per_target_df.to_csv(
    TOP_PER_TARGET_PATH,
    index=False,
)

display(
    top_per_target_df[
        [
            "paper_target",
            "screen_group",
            "drug_name",
            "baseline_target_rank",
            "advanced_target_rank",
            "consensus_target_rank",
            "consensus_target_rank_score",
            "baseline_predicted_pkd",
            "advanced_predicted_pkd",
        ]
    ].round(4)
)

print(
    "Per-target rankings saved to:",
    TOP_PER_TARGET_PATH,
)

## 34. Top 20 unique drug candidates

A single drug can rank highly against more than one AML target.  
For a drug-level repurposing list, each approved drug is represented by its best consensus-ranked target pair.

In [ ]:
best_pair_per_drug_df = (
    screening_results_df
    .sort_values(
        [
            "consensus_global_rank_score",
            "mean_model_predicted_pkd",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        [
            "drug_id",
            "drug_name",
        ],
        as_index=False,
    )
    .first()
)

best_pair_per_drug_df = (
    best_pair_per_drug_df
    .sort_values(
        [
            "consensus_global_rank_score",
            "mean_model_predicted_pkd",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

best_pair_per_drug_df[
    "candidate_rank"
] = np.arange(
    1,
    len(
        best_pair_per_drug_df
    ) + 1,
)

top_unique_drugs_df = (
    best_pair_per_drug_df
    .head(TOP_N)
    .copy()
)

TOP_UNIQUE_DRUGS_PATH = (
    PREDICTIONS_DIR
    / "aml_top20_unique_drug_candidates.csv"
)

top_unique_drugs_df.to_csv(
    TOP_UNIQUE_DRUGS_PATH,
    index=False,
)

display(
    top_unique_drugs_df[
        [
            "candidate_rank",
            "drug_name",
            "paper_target",
            "screen_group",
            "baseline_predicted_pkd",
            "advanced_predicted_pkd",
            "consensus_global_rank_score",
        ]
    ].round(4)
)

print(
    "Unique-drug ranking saved to:",
    TOP_UNIQUE_DRUGS_PATH,
)

# Part J — Screening figures and summary

## 35. Visualise the Top-20 unique candidates

In [ ]:
plot_df = (
    top_unique_drugs_df
    .sort_values(
        "mean_model_predicted_pkd",
        ascending=True,
    )
    .copy()
)

plot_labels = (
    plot_df["drug_name"]
    + " → "
    + plot_df["paper_target"]
)

plt.figure(
    figsize=(10, 8)
)

plt.barh(
    plot_labels,
    plot_df[
        "mean_model_predicted_pkd"
    ],
)

plt.xlabel(
    "Mean model-predicted pKd"
)

plt.ylabel(
    "FDA-approved drug → AML target"
)

plt.title(
    "Top AML Screening Candidates"
)

plt.tight_layout()

TOP_CANDIDATES_FIGURE = (
    RESULTS_DIR
    / "figures"
    / "aml_top20_unique_candidates.png"
)

TOP_CANDIDATES_FIGURE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

plt.savefig(
    TOP_CANDIDATES_FIGURE,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(
    "Figure saved to:",
    TOP_CANDIDATES_FIGURE,
)

## 36. Model agreement among the Top-20 consensus pairs

In [ ]:
agreement_df = (
    consensus_top_df[
        [
            "drug_name",
            "paper_target",
            "baseline_predicted_pkd",
            "advanced_predicted_pkd",
        ]
    ]
    .copy()
)

agreement_df[
    "absolute_model_difference"
] = (
    agreement_df[
        "baseline_predicted_pkd"
    ]
    - agreement_df[
        "advanced_predicted_pkd"
    ]
).abs()

display(
    agreement_df
    .sort_values(
        "absolute_model_difference"
    )
    .round(4)
)

## 37. Final screening summary

In [ ]:
summary_df = pd.DataFrame(
    {
        "item": [
            "AML targets",
            "Primary small-molecule targets",
            "Exploratory targets",
            "FDA ingredients before PubChem/RDKit",
            "Final unique drug structures",
            "Drug-target pairs screened",
            "Selected baseline model",
            "Screening checkpoint split",
        ],
        "value": [
            len(aml_targets_df),
            int(
                (
                    aml_targets_df[
                        "screen_group"
                    ]
                    == "primary"
                ).sum()
            ),
            int(
                (
                    aml_targets_df[
                        "screen_group"
                    ]
                    == "exploratory"
                ).sum()
            ),
            len(fda_ingredients_df),
            len(candidate_drugs_df),
            len(screening_results_df),
            selected_baseline_name,
            SCREENING_SPLIT,
        ],
    }
)

display(summary_df)

print()
print("Saved data")
print("-" * 70)
print("AML targets:", AML_TARGETS_PATH)
print("FDA ingredients:", FDA_INGREDIENTS_PATH)
print("PubChem cache:", PUBCHEM_CACHE_PATH)
print("Candidate drugs:", CANDIDATE_DRUGS_PATH)
print("Structure audit:", CANDIDATE_AUDIT_PATH)

print()
print("Saved screening outputs")
print("-" * 70)
print("All predictions:", ALL_SCREENING_PATH)
print("Baseline Top-20:", BASELINE_TOP_PATH)
print("Advanced Top-20:", ADVANCED_TOP_PATH)
print("Consensus Top-20:", CONSENSUS_TOP_PATH)
print("Primary-target Top-20:", PRIMARY_TOP_PATH)
print("Top-10 per target:", TOP_PER_TARGET_PATH)
print("Top-20 unique drugs:", TOP_UNIQUE_DRUGS_PATH)
print("Figure:", TOP_CANDIDATES_FIGURE)

# Interpretation and limitations

This screening is designed for **hypothesis generation**, not biological or clinical confirmation.

Important points for the dissertation:

1. The seven AML targets are biologically motivated by the target review, while their model input sequences are independently verified as reviewed human UniProt entries.
2. The approved-drug library is based on active ingredients appearing in current RX/OTC products in the FDA Orange Book. It therefore reflects the scope of the Orange Book rather than every therapeutic biologic regulated by FDA.
3. PubChem supplies chemical structures; FDA remains the approval-status source.
4. PubChem structures are standardised with RDKit before molecular-graph construction. Duplicate standardised structures are removed.
5. The DTA models were trained on DAVIS. DAVIS is strongly kinase-oriented, so predictions for non-kinase targets such as BCL2, MEN1, CD123 and CD47 involve greater domain extrapolation than predictions for FLT3.
6. CD123 and CD47 are retained as exploratory targets because many therapeutic strategies directed at these cell-surface proteins are biologics rather than conventional small molecules.
7. Predicted pKd values are model outputs. They are not experimentally measured binding affinities.
8. High-ranked compounds may already have AML indications or known target interactions. These issues are investigated separately in the candidate-validation notebook.

The next notebook should therefore evaluate the highest-ranked candidates against independent evidence and existing clinical/therapeutic knowledge before any repurposing claim is made.

### Notebook completion

This notebook completes the **AML computational screening stage**:

```text
AML target review
      ↓
reviewed UniProt proteins
      +
FDA-approved active ingredients
      ↓
PubChem + RDKit structures
      ↓
best DAVIS baseline + advanced model
      ↓
predicted drug-target affinities
      ↓
model-specific and consensus rankings
      ↓
Top candidate drug-target pairs
      ↓
Top unique approved-drug candidates
```

The final ranked outputs are intentionally passed to Notebook 10 for independent candidate validation.